# 🧊 Neural Gesture Mannequin — 2D-Anchored 3D Volumetric Mesh Reconstruction
**Author:** Muhammad Aliffandy (Apple Developer Academy & PukulEnam AI/ML Engineer)  
**Architecture:** Two-Stage Hybrid (Stage 1: 2D Keypoint Geometry Anchor [PCK 84.7%] + Stage 2: 3D Volumetric Mannequin Mesh Synthesis)  

### 🌟 Why This Solves the Pose-Matching Problem:
1. **Exact 2D Pose Grounding:** Uses your DGX-trained model (`volumetric_posenet_best.pth`) to lock the exact 2D joint locations (hands on hips, bent elbows, A-frame legs) directly from the photo.
2. **3D Metric Kinematic Lifting:** Lifts the 2D keypoints into 3D metric coordinates $(X, Y, Z)$ using biomechanical bone length constraints.
3. **True Volumetric Mesh Synthesis:** Constructs 3D anatomical cylinders, torso prism, and head ellipsoid parameterized by your model's muscle volume radii.
4. **Multi-Angle Orthogonal Views:** Simultaneously renders Front (0° matching photo!), Side Profile (90°), and Bird-Eye Perspective (45°).
5. **Interactive 3D WebGL Canvas:** Rotate the mannequin 360° freely in your browser.
6. **Export to 3D File:** Generates `.OBJ` ready for Blender, Maya, SceneKit, or iPadOS PencilKit integration!

## 🛠️ Step 1: Clone Repository & Install Dependencies

In [ ]:
# 1. Clone repository
!rm -rf hp-tra
!git clone https://github.com/MuhammadAliffandy/hp-tra.git
%cd hp-tra

# 2. Install lightweight dependencies
!pip install -q pycocotools torchvision opencv-python matplotlib Pillow plotly trimesh

## ☁️ Step 2: Mount Google Drive & Load Trained Model (`volumetric_posenet_best.pth`)

In [ ]:
from google.colab import drive
import os
import torch
from src.model import MultiTaskVolumetricPoseNet

# Mount Drive
drive.mount('/content/drive')

# Path to your high-accuracy model trained on 114,554 COCO images (PCK 84.7%)
MODEL_PATH = "/content/drive/MyDrive/Neural_Gesture_Mannequin_Models/volumetric_posenet_best.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Compute Device: {device}")

# Instantiate Model (MobileNetV3-Large Multi-Task PoseNet)
model = MultiTaskVolumetricPoseNet(backbone_name="mobilenet_v3_large", pretrained=False).to(device)

if os.path.exists(MODEL_PATH):
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    if "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        val_pck = checkpoint.get("val_pck", 84.7)
        val_err = checkpoint.get("val_err", 15.37)
        print(f"✅ High-Accuracy Model successfully loaded from Google Drive!")
        print(f"   - Best Val PCK@0.1     : {val_pck:.1f} %")
        print(f"   - Best Val Pixel Error : {val_err:.2f} px")
    else:
        model.load_state_dict(checkpoint)
        print("✅ Model state dict loaded successfully!")
else:
    print(f"⚠️ Checkpoint not found at: {MODEL_PATH}")
    print("Please verify that rclone copy ... xxdrive:Neural_Gesture_Mannequin_Models/ has completed.")

model.eval()

## 📸 Step 3: Upload Your Photo (or Provide URL)

In [ ]:
from google.colab import files
from PIL import Image
import urllib.request

# Choose input mode:
# Option A: Upload from your laptop
# Option B: Use an image URL from the web

IMAGE_URL = ""  # Leave empty if you want to upload a file from your computer
TARGET_IMAGE_PATH = "test_action.jpg"

if IMAGE_URL.strip() != "":
    print(f"🌐 Downloading image from URL: {IMAGE_URL}")
    urllib.request.urlretrieve(IMAGE_URL, TARGET_IMAGE_PATH)
    print("✅ Download complete!")
else:
    print("📁 Click 'Choose Files' below to upload your photo:")
    uploaded = files.upload()
    if uploaded:
        TARGET_IMAGE_PATH = list(uploaded.keys())[0]
        print(f"✅ Uploaded: {TARGET_IMAGE_PATH}")
    else:
        print("No file selected, using default sample.")

## 🚀 Step 4: Run 2D-Anchored Kinematic Lifting & Synthesize 3D Mesh
Detects precise 2D keypoints from your photo, lifts them into 3D metric space, and inflates volumetric anatomical primitives.

In [ ]:
import sys
for mod in list(sys.modules.keys()):
    if 'src.mesh_generator' in mod:
        del sys.modules[mod]

import numpy as np
import torchvision.transforms as T
from PIL import Image
from src.mesh_generator import lift_2d_to_3d_joints, build_volumetric_mannequin, export_to_obj

# 1. Load and Preprocess Image
raw_pil = Image.open(TARGET_IMAGE_PATH).convert("RGB")
resized_pil = raw_pil.resize((256, 256), Image.Resampling.BILINEAR)

transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
inp_tensor = transform(raw_pil).unsqueeze(0).to(device)

# 2. Forward Pass through High-Accuracy Model (PCK 84.7%)
with torch.no_grad():
    pred_coords, pred_volume, pred_vis_logits = model(inp_tensor)
    coords_np = pred_coords[0].cpu().numpy()             # [17, 2] in [0, 1]
    volume_np = pred_volume[0].cpu().numpy()             # [6] in [0, 1]
    vis_np = torch.sigmoid(pred_vis_logits[0]).cpu().numpy() # [17] confidence

print("🔮 2D Pose & Volume Predictions:")
print(f"   - 17 Joint Coordinates : {coords_np.shape} (anchored directly to photo!)")
print(f"   - 6 Muscle Radii (Vol) : {volume_np.round(3)}")
print(f"   - Mean Detection Conf  : {vis_np.mean():.2f}")

# 3. Lift 2D Coordinates into 3D Metric Coordinates (Meters)
joints_3d = lift_2d_to_3d_joints(coords_np, volume_np)

# 4. Synthesize 3D Volumetric Mannequin Mesh (Wrapped around joints)
mesh_verts, mesh_faces = build_volumetric_mannequin(joints_3d, volume_np)

# 5. Export to Standard Wavefront .OBJ Format
OUTPUT_OBJ_PATH = "mannequin_reconstructed.obj"
export_to_obj(mesh_verts, mesh_faces, OUTPUT_OBJ_PATH)
print(f"🎉 SUCCESS! 3D Mesh synthesized ({len(mesh_verts)} vertices, {len(mesh_faces)} faces)")
print(f"💾 Saved Wavefront .OBJ mesh to: {OUTPUT_OBJ_PATH}")

## 📐 Step 5: Multi-Angle Orthographic & Perspective Showcase
Shows the **Front View (0°)** perfectly matching your photo's pose (hands on hips, bent elbows, wide stance), alongside **Side Profile (90°)** and **Bird-Eye Perspective (45°)**!

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# Coordinate Mapping for Matplotlib 3D Viewers:
# mesh_verts[:, 0] = lateral (X, right is +)
# mesh_verts[:, 1] = vertical height (Y, up is +)
# mesh_verts[:, 2] = depth (Z, forward is +)
v_plot = np.zeros_like(mesh_verts)
v_plot[:, 0] = mesh_verts[:, 0]        # X: lateral
v_plot[:, 1] = mesh_verts[:, 2]        # Y: depth
v_plot[:, 2] = mesh_verts[:, 1]        # Z: height (up is +, head at top, feet at bottom!)

fig = plt.figure(figsize=(22, 6))

# Panel 1: Original Input Photo
ax0 = fig.add_subplot(141)
ax0.imshow(resized_pil)
ax0.set_title("[1] Input Photo", fontsize=13, fontweight='bold', pad=10)
ax0.axis("off")

# Panel 2: Front View (0° — Matches photo pose!)
ax1 = fig.add_subplot(142, projection='3d')
ax1.view_init(elev=5, azim=-90)
m1 = Poly3DCollection(v_plot[mesh_faces], alpha=0.92, edgecolor='#222222', linewidths=0.25)
m1.set_facecolor([0.88, 0.78, 0.72])
ax1.add_collection3d(m1)
ax1.set_xlim(-0.8, 0.8); ax1.set_ylim(-0.8, 0.8); ax1.set_zlim(-1.0, 0.8)
ax1.axis('off')
ax1.set_title("[2] Front View (0°)\n(Matches Photo Pose!)", fontsize=12, fontweight='bold', pad=10)

# Panel 3: Side View (90° Profile)
ax2 = fig.add_subplot(143, projection='3d')
ax2.view_init(elev=5, azim=0)
m2 = Poly3DCollection(v_plot[mesh_faces], alpha=0.92, edgecolor='#222222', linewidths=0.25)
m2.set_facecolor([0.88, 0.78, 0.72])
ax2.add_collection3d(m2)
ax2.set_xlim(-0.8, 0.8); ax2.set_ylim(-0.8, 0.8); ax2.set_zlim(-1.0, 0.8)
ax2.axis('off')
ax2.set_title("[3] Side Profile (90°)\n(Shows 3D Depth)", fontsize=12, fontweight='bold', pad=10)

# Panel 4: Bird-Eye Perspective (45°)
ax3 = fig.add_subplot(144, projection='3d')
ax3.view_init(elev=35, azim=-60)
m3 = Poly3DCollection(v_plot[mesh_faces], alpha=0.92, edgecolor='#222222', linewidths=0.25)
m3.set_facecolor([0.88, 0.78, 0.72])
ax3.add_collection3d(m3)
ax3.set_xlim(-0.8, 0.8); ax3.set_ylim(-0.8, 0.8); ax3.set_zlim(-1.0, 0.8)
ax3.axis('off')
ax3.set_title("[4] Bird-Eye Perspective (45°)\n(Volumetric Geometry)", fontsize=12, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig("multi_angle_showcase.png", dpi=200, bbox_inches="tight")
plt.show()
print("🖼️ Multi-angle showcase saved to: multi_angle_showcase.png")

## 🌐 Step 6: Interactive 3D WebGL Canvas (Click & Rotate 360°!)
Use your mouse to **click, drag, rotate in 360°, and zoom in/out** right inside this notebook!

In [ ]:
import plotly.graph_objects as go

# Create interactive 3D WebGL Mesh
fig = go.Figure(data=[
    go.Mesh3d(
        x=v_plot[:, 0],
        y=v_plot[:, 1],
        z=v_plot[:, 2],
        i=mesh_faces[:, 0],
        j=mesh_faces[:, 1],
        k=mesh_faces[:, 2],
        color='#E0C4B0',
        opacity=0.95,
        flatshading=True,
        lighting=dict(ambient=0.6, diffuse=0.8, specular=0.4, roughness=0.5)
    )
])

fig.update_layout(
    title="<b>🎨 Neural Gesture Mannequin — Interactive 3D Canvas (Drag to Rotate 360°)</b>",
    scene=dict(
        xaxis=dict(showbackground=False, showgrid=False, zeroline=False, visible=False),
        yaxis=dict(showbackground=False, showgrid=False, zeroline=False, visible=False),
        zaxis=dict(showbackground=False, showgrid=False, zeroline=False, visible=False),
        aspectmode='data'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    width=800,
    height=600
)

fig.show()

## 💾 Step 7: Download 3D Mesh (`.OBJ`) for Blender, Maya & SceneKit

In [ ]:
from google.colab import files
import os

if os.path.exists(OUTPUT_OBJ_PATH):
    print(f"📦 Downloading {OUTPUT_OBJ_PATH} to your computer...")
    files.download(OUTPUT_OBJ_PATH)
    print("✅ You can now open this .OBJ file in Blender, Maya, Clip Studio Paint, or Mac QuickLook!")
else:
    print(f"File not found: {OUTPUT_OBJ_PATH}")